In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F 

In [2]:
# read data
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

In [ ]:
# hyperparameters
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
dropout = 0.2
device = 'cuda'

eval_iters = 200

n_embd = 384
num_heads = 6
head_size = n_embd // num_heads

### 多头注意力

In [ ]:
#一个head
class Head(nn.Module):
    def __init__(self,):
        #head_size = C / num_size：一个头的特征数 
        super().__init__()
        self.q = nn.Linear(n_embd, head_size, bias = False)
        self.k = nn.Linear(n_embd, head_size, bias = False)
        self.v = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        #x是embedding后的向量矩阵
        #x(B, T, C) @ wT(C, head_size)-> q,k,v(B, T, head_size)
        B, T, C = x.shape
        q = self.q(x)     
        k = self.k(x)
        v = self.v(x) 
        wei = (q @ k.transpose(-1, -2)) / math.sqrt(head_size)   #(B, T, head_size) @ (B, head_size, T) ---> (B, T, T)
        #加掩码
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1) 
        wei = self.dropout(wei)
        out = wei @ v
        return out 

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(num_heads)])
        '''
        # 等价展开
        self.head0 = Head(head_size)
        self.head1 = Head(head_size)
        self.head2 = Head(head_size)
        self.head3 = Head(head_size)
        self.heads = nn.ModuleList([self.head0, self.head1, self.head2, self.head3])
        '''
        self.proj = nn.Linear(n_embd, n_embd) #投影层
        self.dropout = nn.Dropout(dropout)


    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.proj(out)
        out = self.dropout(out)
        return out
    

### 前馈网络

In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Seqiential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

        '''
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        '''

    def forward(self, x):
        '''
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
        '''
        return self.net(x)

### 构建模块

In [7]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHeadAttention()
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        '''
        LayerNorm内部有:
        weight（γ，缩放参数）和bias（β，平移参数）
        训练过程中，ln1和ln2朝着不同方向更新
        '''

    def forward(self, x):  #残差链接
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x)) 
        return x

In [ ]:
# Bigram Model
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)  #可学习位置编码，不用正弦位置编码
        self.blocks = nn.Sequential(
            Block(),
            Block(),
            Block(),
            nn.LayerNorm(n_embd)
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)                   # 输出每个 token 的打分
        

    def forward(self, x, targets=None):
        B, T = x.shape

        tok_emb = self.token_embedding_table(x)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device)) #模型内部新建的张量，都要显式指定设备
        x = tok_emb + pos_emb
        x = self.blocks(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        for _ in range(max_new_tokens):
            x_cond = x[:, -block_size:]
            logits, loss = self(x_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            x_next = torch.multinomial(probs, num_samples=1)
            x = torch.cat((x, x_next), dim=1)
        return x

In [9]:
# tokenizer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# train test split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



model = BigramLanguageModel(vocab_size)
m = model.to(device)

# optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

# training loop
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=100)[0].tolist()))
# %%


step 0: train loss 4.5788, val loss 4.5782
step 300: train loss 2.3278, val loss 2.3379
step 600: train loss 2.2418, val loss 2.2758
step 900: train loss 2.1852, val loss 2.2348
step 1200: train loss 2.1458, val loss 2.1894
step 1500: train loss 2.0910, val loss 2.1866
step 1800: train loss 2.0872, val loss 2.1714
step 2100: train loss 2.0709, val loss 2.1808
step 2400: train loss 2.0602, val loss 2.1607
step 2700: train loss 2.0507, val loss 2.1386

With by that fyet bristeas, touch,
My non. My iWh By,
E way.
I

First AS Cictuss suque'
Wore yet fre
